# Crack detection: run the week-1 code in Colab

Runs **Baseline 1 (Canny)** and **YOLOv8n** on the team's merged dataset and logs every run to `results/experiment_log.csv`.

**Uploading once is enough.** Your code zip and dataset zip are stored in your Google Drive (`MyDrive/crack-detection/`). On later runs the notebook finds them there and skips the upload. Results are copied back to Drive after each run.

**Before you start:** Runtime > Change runtime type > **T4 GPU** (only the YOLO step needs it).

Run the cells top to bottom.

## 1. Connect Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/crack-detection")

if not DRIVE.exists():
    raise FileNotFoundError(
        f"{DRIVE} not found. If this is a shared folder, add a shortcut to it "
        "in My Drive (Drive web -> Shared with me -> right-click -> Organize -> Add shortcut)."
    )

(DRIVE / "uploads").mkdir(exist_ok=True)
(DRIVE / "results").mkdir(exist_ok=True)

## 2. Code: upload `crack-detection-week1.zip` (first time only)

If the zip is already in `MyDrive/crack-detection/uploads/`, it is reused and no upload is asked for.
Set `REUPLOAD_CODE = True` if you made a new version of the code and want to replace it.

In [ ]:
REUPLOAD_CODE = False

code_zip = DRIVE / "uploads" / "crack-detection-code.zip"
if REUPLOAD_CODE or not code_zip.exists():
    from google.colab import files
    print("Select crack-detection-week1.zip")
    up = files.upload()
    name = next(iter(up))
    shutil.move(name, code_zip)          # stored in Drive for next time
    print("Saved to Drive:", code_zip)
else:
    print("Using code zip from Drive:", code_zip)

# fresh local copy of the code (keeps a previous data/ folder if there is one)
tmp = Path("/content/_code_tmp")
shutil.rmtree(tmp, ignore_errors=True)
with zipfile.ZipFile(code_zip) as z:
    z.extractall(tmp)
src = next(p.parent for p in tmp.rglob("requirements.txt") if (p.parent / "models").is_dir())
WORK.mkdir(exist_ok=True)
for item in src.iterdir():
    if item.name in ("data", "results"):
        continue
    dest = WORK / item.name
    if dest.is_dir():
        shutil.rmtree(dest)
    elif dest.exists():
        dest.unlink()
    shutil.move(str(item), dest)
(WORK / "data").mkdir(exist_ok=True)
(WORK / "results").mkdir(exist_ok=True)
os.chdir(WORK)
print("Working in", Path.cwd())
print(sorted(p.name for p in WORK.iterdir()))

In [ ]:
!pip install -q -r requirements.txt
import ultralytics; print("ultralytics", ultralytics.__version__)

### Patch: updated model scripts

These two cells overwrite `models/train_yolo.py` (fixes the "best.pt not found" crash with newer Ultralytics and logs F1) and add `models/eval_yolo.py` (scores saved weights). They run after the code zip is unpacked, so the fix applies every session. Once these changes are in your code zip, you can delete the two cells.

In [ ]:
%%writefile models/train_yolo.py
"""Train a YOLOv8 detector (Ultralytics) and log its test-set scores.

Plain English: start from a model already trained on everyday photos
(yolov8n.pt), then keep training it on our steel images so it learns defects.

Usage:
    python -m models.train_yolo --config configs/yolo_merged_v8n.yaml --device 0
"""
import argparse
from pathlib import Path

from ultralytics import YOLO

from models.utils import (load_config, log_experiment, resolved_data_yaml,
                          save_json, set_seed)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/yolo_merged_v8n.yaml")
    ap.add_argument("--device", default=None, help="e.g. 0 for GPU, cpu")
    args = ap.parse_args()
    cfg = load_config(args.config)
    set_seed(cfg["seed"])

    data = resolved_data_yaml(cfg["data"])
    model = YOLO(cfg["model"])
    model.train(
        data=data, epochs=cfg["epochs"], imgsz=cfg["imgsz"],
        batch=cfg["batch"], patience=cfg.get("patience", 50),
        workers=cfg.get("workers", 2), seed=cfg["seed"], deterministic=True,
        # absolute path: newer Ultralytics adds a "detect/" folder to relative ones
        project=str(Path(cfg["project"]).resolve()), name=cfg["run_name"],
        exist_ok=True, device=args.device,
    )

    # Ask the trainer where it really saved the weights, then score on test.
    best_path = Path(model.trainer.save_dir) / "weights" / "best.pt"
    print("Scoring", best_path)
    best = YOLO(str(best_path))
    m = best.val(data=data, split=cfg["eval_split"],
                 imgsz=cfg["imgsz"], device=args.device)
    p, r = float(m.box.mp), float(m.box.mr)
    metrics = {
        "split": cfg["eval_split"], "precision": p, "recall": r,
        "f1": 2 * p * r / (p + r) if p + r else 0.0,
        "map50": float(m.box.map50), "map50_95": float(m.box.map),
        "seed": cfg["seed"],
    }
    print(metrics)
    save_json(metrics, f"results/{cfg['run_name']}.json")
    log_experiment(cfg["run_name"], args.config, metrics)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile models/eval_yolo.py
"""Score an already-trained YOLO checkpoint on a split and log it.

Use this when training finished but the final scoring step did not run.

Usage:
    python -m models.eval_yolo --weights path/to/best.pt --run-name yolov8n_merged_v1 --split test --device 0
"""
import argparse

from ultralytics import YOLO

from models.utils import log_experiment, resolved_data_yaml, save_json


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--run-name", required=True)
    ap.add_argument("--data", default="data/merged/data.yaml")
    ap.add_argument("--split", default="test")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--device", default=None)
    args = ap.parse_args()

    m = YOLO(args.weights).val(data=resolved_data_yaml(args.data), split=args.split,
                               imgsz=args.imgsz, device=args.device)
    p, r = float(m.box.mp), float(m.box.mr)
    metrics = {
        "split": args.split, "precision": p, "recall": r,
        "f1": 2 * p * r / (p + r) if p + r else 0.0,
        "map50": float(m.box.map50), "map50_95": float(m.box.map),
    }
    print(metrics)
    save_json(metrics, f"results/{args.run_name}_{args.split}.json")
    log_experiment(args.run_name, "eval_yolo", metrics)


if __name__ == "__main__":
    main()

## 3. Dataset: upload the merged dataset zip (first time only)

Zip Millie's **`data/merged/`** folder (it must contain `data.yaml`, `train.txt`, `val.txt`, `test.txt`, `images/`, `labels/`) and upload it once.

Big zip? Browser uploads can be slow. Alternative: put the zip straight into the Drive folder **`My Drive/crack-detection/uploads/`** from drive.google.com, name it anything ending in `.zip` that is not `crack-detection-code.zip`, then run this cell and it will be found automatically.

Set `REUPLOAD_DATA = True` when the team ships a new frozen version.

In [ ]:
REUPLOAD_DATA = False

def has_data_yaml(zp):
    try:
        with zipfile.ZipFile(zp) as z:
            return any(n.rsplit("/", 1)[-1] in ("data.yaml", "data.yml") and "__MACOSX" not in n for n in z.namelist())
    except zipfile.BadZipFile:
        return False

def find_data_zip():
    zs = sorted((p for p in (DRIVE / "uploads").glob("*.zip") if p.name != "crack-detection-code.zip"),
                key=lambda p: p.stat().st_mtime, reverse=True)
    for p in zs:                      # newest zip that really holds a dataset (has data.yaml)
        if has_data_yaml(p):
            return p
        print(f"Skipping {p.name}: no data.yaml inside (this looks like code, not a dataset)")
    return None

data_zip = find_data_zip()
if REUPLOAD_DATA or data_zip is None:
    from google.colab import files
    print("Select the DATASET .zip (the one that contains data.yaml, images/ and labels/)")
    up = files.upload()
    name = next(iter(up))
    data_zip = DRIVE / "uploads" / name
    shutil.move(name, data_zip)
    if not has_data_yaml(data_zip):
        raise FileNotFoundError(f"{name} has no data.yaml inside, so it is not a dataset. "
                                "It is probably the code zip. Delete it from MyDrive/crack-detection/uploads/ "
                                "and upload the zip of the merged dataset (or a Roboflow YOLOv8 export of NEU-DET).")
    print("Saved to Drive:", data_zip)
else:
    print("Using dataset zip from Drive:", data_zip.name)

MERGED = WORK / "data" / "merged"
shutil.rmtree(MERGED, ignore_errors=True)
tmp = Path("/content/_data_tmp")
shutil.rmtree(tmp, ignore_errors=True)
with zipfile.ZipFile(data_zip) as z:
    z.extractall(tmp)

# find data.yaml wherever the zip put it (ignoring macOS junk), prefer the one next to train.txt/images
cands = [p for p in tmp.rglob("*") if p.name in ("data.yaml", "data.yml")
         and "__MACOSX" not in p.parts and not p.name.startswith("._")]
if not cands:
    print("No data.yaml found in the zip. Top of its contents:")
    for p in sorted(tmp.rglob("*"))[:30]:
        print("  ", p.relative_to(tmp))
    raise FileNotFoundError("The dataset zip has no data.yaml. Zip Millie's data/merged folder (it must contain data.yaml).")
cands.sort(key=lambda p: (not ((p.parent / "train.txt").exists() or (p.parent / "images").exists()), len(p.parts)))
root = cands[0].parent
MERGED.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(root, MERGED)
if not (MERGED / "data.yaml").exists():          # data.yml -> data.yaml
    shutil.copy(cands[0], MERGED / "data.yaml")
shutil.rmtree(tmp, ignore_errors=True)
# Roboflow-style export (train/valid/test folders, no train.txt)? Rewrite data.yaml so paths resolve.
import yaml
_c = yaml.safe_load(open(MERGED / "data.yaml"))
if not (MERGED / "train.txt").exists():
    def _dir(*names):
        for n in names:
            if (MERGED / n / "images").is_dir():
                return f"{n}/images"
    fixed = {"path": str(MERGED), "train": _dir("train"), "val": _dir("valid", "val"), "test": _dir("test"),
             "nc": _c.get("nc"), "names": _c.get("names")}
    fixed = {k: v for k, v in fixed.items() if v is not None}
    yaml.safe_dump(fixed, open(MERGED / "data.yaml", "w"))
    print("Roboflow-style layout detected, data.yaml rewritten:", fixed)
    if "test" not in fixed:
        print("WARNING: no test split in this export; the scripts evaluate on 'test'. Set eval_split to val or re-export with a test split.")
print("Dataset ready at", MERGED)
print(sorted(p.name for p in MERGED.iterdir()))

### Quick check of the dataset

In [ ]:
import yaml
from models.utils import list_images, label_path, read_yolo_labels
import cv2
from collections import Counter

assert (MERGED / "data.yaml").exists(), "Run the dataset upload cell above first (section 3)."
cfg = yaml.safe_load(open(MERGED / "data.yaml"))
print("data.yaml:", cfg, "\n")

for split in ["train", "val", "test"]:
    imgs = list_images(str(MERGED / "data.yaml"), split)
    with_box = sum(1 for p in imgs if label_path(p).exists() and label_path(p).read_text().strip())
    print(f"{split:5s} {len(imgs):5d} images, {with_box:5d} with at least one crack box, {len(imgs)-with_box:5d} crack-free")

card = MERGED / "DATA_CARD.md"
if card.exists():
    print("\n--- DATA_CARD.md (first 40 lines) ---")
    print("\n".join(card.read_text().splitlines()[:40]))

In [ ]:
# look at 6 test images with their ground-truth boxes (green)
import matplotlib.pyplot as plt, random
random.seed(0)
imgs = [p for p in list_images(str(MERGED / "data.yaml"), "test") if label_path(p).exists() and label_path(p).read_text().strip()]
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, p in zip(axes.ravel(), random.sample(imgs, min(6, len(imgs)))):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    for _, x1, y1, x2, y2 in read_yolo_labels(label_path(p), w, h):
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
    ax.imshow(im); ax.set_title(p.parent.name + "/" + p.name, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Keep results across sessions

`restore_results()` copies earlier results from Drive so the experiment log keeps growing instead of restarting.
`save_results()` copies everything back to Drive. It is called automatically after each run below.

In [ ]:
def restore_results():
    for p in (DRIVE / "results").glob("*"):
        dest = WORK / "results" / p.name
        if p.is_dir():
            shutil.copytree(p, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(p, dest)

def save_results():
    for p in (WORK / "results").glob("*"):
        dest = DRIVE / "results" / p.name
        if p.is_dir():
            shutil.copytree(p, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(p, dest)
    if (WORK / "runs").exists():
        shutil.copytree(WORK / "runs", DRIVE / "results" / "runs", dirs_exist_ok=True)
    print("Results saved to", DRIVE / "results")

restore_results()
print(sorted(p.name for p in (WORK / "results").iterdir()))

## 5. Baseline 1: Canny edge detection

No GPU needed. Tunes on `val`, reports once on `test` (IoU 0.5). It tries 54 setting combinations, so it can take a few minutes.

In [ ]:
!python -m models.baseline_canny --config configs/baseline_canny.yaml
save_results()

## 6. YOLOv8n detector

Needs the GPU runtime. Set `EPOCHS = 3` first for a quick test that the pipeline works, then `50` for the real run.
The run name is what appears in the experiment log, so change it when you change a setting.

In [ ]:
EPOCHS   = 50                 # use 3 for a quick smoke test
RUN_NAME = "yolov8n_merged_v1"
IMGSZ, BATCH, PATIENCE, SEED = 640, 16, 15, 0

base = yaml.safe_load(open("configs/yolo_merged_v8n.yaml"))
base.update(run_name=RUN_NAME, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, seed=SEED)
yaml.safe_dump(base, open(f"configs/{RUN_NAME}.yaml", "w"))
print(base)

In [ ]:
!python -m models.train_yolo --config configs/{RUN_NAME}.yaml --device 0
save_results()

### 6b. Score saved weights on the test split

Training saves `best.pt` and then scores it on `test`. If that last step failed, or you restarted the session, run this cell: it finds the newest `best.pt` for `RUN_NAME` and adds a **test** row to the experiment log. No retraining needed.

In [ ]:
import glob, os
try:
    run_name = RUN_NAME
except NameError:
    run_name = "yolov8n_merged_v1"

found = sorted(glob.glob(f"**/{run_name}/weights/best.pt", recursive=True), key=os.path.getmtime)
assert found, f"No best.pt found for {run_name}. Run section 6 first (or restore_results())."
WEIGHTS = found[-1]
print("Scoring", WEIGHTS)
!python -m models.eval_yolo --weights {WEIGHTS} --run-name {run_name} --split test --device 0
save_results()

## 7. Results

In [ ]:
import pandas as pd
log = pd.read_csv("results/experiment_log.csv")
cols = [c for c in ["timestamp", "run", "split", "precision", "recall", "f1", "map50", "map50_95", "seed", "git_commit"] if c in log.columns]
display(log[cols])

In [ ]:
# training curves and a few predictions from the YOLO run
import glob, os
from IPython.display import Image, display

try:
    run_name = RUN_NAME
except NameError:
    run_name = "yolov8n_merged_v1"
    print(f"RUN_NAME not defined in this session, using: {run_name}")

# Newer Ultralytics saves under runs/detect/..., and restored results sit in results/runs/...
hits = sorted(glob.glob(f"**/{run_name}/results.png", recursive=True), key=os.path.getmtime)
if not hits:
    print("No results.png found for", run_name)
else:
    run_dir = os.path.dirname(hits[-1])
    print("Using", run_dir)
    display(Image(hits[-1], width=900))
    for f in sorted(glob.glob(f"{run_dir}/val_batch*_pred.jpg"))[:2]:
        display(Image(f, width=900))


## 8. Download results for the slides

Everything is also in Drive (`MyDrive/crack-detection/results/`). This cell bundles the log and score files for a quick download, so you can send them back to Claude to fill the results slide.

In [ ]:
save_results()
out = shutil.make_archive("/content/crack_results", "zip", WORK / "results")
from google.colab import files
files.download(out)